# Controlled Decoding: шаг 2, обучение prefix scorer'а (CD-FUDGE)

Prefix scorer V(x, yᵗ) предсказывает итоговый reward r(x, y) по началу ответа yᵗ. В CD-FUDGE
таргет для каждого префикса один и тот же — reward полного ответа, сэмплированного из π_ref:

L(θ) = E_{x, y~π_ref} Σ_t (V_θ(x, yᵗ) − r(x, y))²

Архитектура, как в статье: та же модель, что и π_ref (`Qwen2.5-0.5B-Instruct`), с линейной головой
hidden → 1 на каждой позиции (`AutoModelForTokenClassification`, `num_labels=1`). Значение на позиции
`P−1+t` (P — длина промпта) относится к префиксу из t токенов ответа: t=0 значит пустой ответ, t=T полный.

Reward стандартизуем по train: блочному CD нужен только argmax по кандидатам, масштаб не важен,
а оптимизации так проще.

**Главная метрика — pairwise accuracy внутри промпта:** для двух ответов на один промпт V по первым
t токенам должен правильно угадать, какой ответ получит больший reward. Именно такой выбор делает
блочный CD. Accuracy@32 показывает, насколько осмысленно CD выбирает блоки при M=32.

**Вход:** датасет с выходом `01_rollouts` (папка `rollouts/` с `train.parquet` и `config.json`), подключённый
через Add Input.

**Промежуточные сохранения:** каждые `SAVE_EVERY` шагов полный чекпоинт (веса, оптимизатор, scheduler,
GradScaler, шаг) пишется в `scorer/ckpt`. Если упало ядро, Restart и Run All продолжат с чекпоинта.
`/kaggle/working` живёт только в пределах сессии. Чтобы продолжить в **новой** сессии, подключи
предыдущую версию этого ноутбука как input: чекпоинт найдётся в `/kaggle/input/**/scorer/ckpt`.

**Kaggle:** GPU T4 x2 (используется одна GPU), Internet On.

In [ ]:
import os, json, glob, time, shutil, math
from collections import defaultdict
from itertools import combinations
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoTokenizer, AutoModelForTokenClassification, get_cosine_schedule_with_warmup

print(torch.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
DEBUG = True  # короткий прогон на 256 промптах; для полного прогона поставить False

INIT_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
ROLLOUTS = None  # None: найти rollouts/train.parquet в /kaggle/input автоматически

VAL_FRAC = 0.05        # доля промптов (не ответов) в валидации
EPOCHS = 1
MICRO_BS = 8
ACCUM = 4              # эффективный батч 32 последовательности
LR = 1e-5              # backbone
HEAD_LR = 1e-4         # голова инициализирована случайно
WARMUP = 2 if DEBUG else 30
MAX_GRAD_NORM = 1.0
EVAL_EVERY = 5 if DEBUG else 100
SAVE_EVERY = 5 if DEBUG else 100
LOG_EVERY = 10
EVAL_PREFIXES = [8, 16, 32, 64, 128, 256]  # t токенов ответа; t ≥ T означает полный ответ
PAIR_MIN_GAP = 0.5     # в единицах сырого reward: пары с почти равным reward не считаем
SEED = 0
DEVICE = "cuda:0"

WORK = "/kaggle/working" if os.path.exists("/kaggle") else "."
OUT = f"{WORK}/scorer_debug" if DEBUG else f"{WORK}/scorer"
CKPT = f"{OUT}/ckpt"
FINAL = f"{OUT}/final"
METRICS = f"{OUT}/metrics.jsonl"
os.makedirs(OUT, exist_ok=True)

## Данные

In [ ]:
if ROLLOUTS is None:
    # train.parquet рядом с config.json: так находится и папка rollouts/ из Output, и плоский датасет из API
    found = [f for f in sorted(glob.glob("/kaggle/input/**/train.parquet", recursive=True)) + sorted(glob.glob("rollouts/train.parquet"))
             if os.path.exists(os.path.join(os.path.dirname(f), "config.json")) and "/shards/" not in f and "rollouts_debug" not in f]
    assert len(found) == 1, f"Нужен ровно один rollouts/train.parquet, найдено: {found}. Укажи ROLLOUTS явно."
    ROLLOUTS = os.path.dirname(found[0])
rollout_cfg = json.load(open(f"{ROLLOUTS}/config.json"))
print(ROLLOUTS, json.dumps(rollout_cfg, indent=1))
assert DEBUG or not rollout_cfg["debug"], "Подключены отладочные роллауты"
assert rollout_cfg["base_model"] == INIT_MODEL, "scorer должен использовать тот же токенизатор, что и π_ref"

df = pd.read_parquet(f"{ROLLOUTS}/train.parquet")
if DEBUG:
    df = df[df.pid < 256]

pids = np.random.default_rng(SEED).permutation(df.pid.unique())
val_pids = set(pids[: max(1, int(len(pids) * VAL_FRAC))].tolist())
is_val = df.pid.isin(val_pids)
R_MEAN, R_STD = float(df.reward[~is_val].mean()), float(df.reward[~is_val].std())

def to_examples(d):
    return [{"ids": np.concatenate([r.prompt_ids, r.response_ids]).astype(np.int64), "p": len(r.prompt_ids),
             "target": (r.reward - R_MEAN) / R_STD, "reward": r.reward, "pid": r.pid}
            for r in d.itertuples()]

train_ex, val_ex = to_examples(df[~is_val]), to_examples(df[is_val])
print(f"train {len(train_ex)} seq, val {len(val_ex)} seq ({len(val_pids)} промптов); reward mean {R_MEAN:.2f}, std {R_STD:.2f}")

tok = AutoTokenizer.from_pretrained(INIT_MODEL)

def collate(items):
    L = max(len(x["ids"]) for x in items)
    ids = torch.full((len(items), L), tok.pad_token_id, dtype=torch.long)
    att = torch.zeros((len(items), L), dtype=torch.long)
    mask = torch.zeros((len(items), L), dtype=torch.bool)
    for i, x in enumerate(items):
        n = len(x["ids"])
        ids[i, :n] = torch.from_numpy(x["ids"])
        att[i, :n] = 1
        mask[i, x["p"] - 1:n] = True  # префиксы ответа длины 0..T
    tgt = torch.tensor([x["target"] for x in items], dtype=torch.float32)
    return {"input_ids": ids, "attention_mask": att, "mask": mask, "target": tgt}

def make_schedule():
    # Детерминированный порядок микробатчей: при возобновлении пропускаем уже пройденные.
    # Внутри «мегабатча» сортируем по длине, чтобы было меньше паддинга, а потом перемешиваем сами микробатчи.
    lengths = [len(x["ids"]) for x in train_ex]
    sched = []
    for ep in range(EPOCHS):
        rng = np.random.default_rng(SEED + ep)
        idx = rng.permutation(len(train_ex))
        batches = []
        for i in range(0, len(idx), MICRO_BS * 64):
            chunk = sorted(idx[i:i + MICRO_BS * 64], key=lambda j: lengths[j])
            batches += [chunk[k:k + MICRO_BS] for k in range(0, len(chunk), MICRO_BS)]
        sched += [batches[i] for i in rng.permutation(len(batches))]
    return sched

schedule = make_schedule()
TOTAL_STEPS = len(schedule) // ACCUM
print("optimizer steps:", TOTAL_STEPS)

## Модель и чекпоинты

Веса в fp32, forward в fp16 через autocast с GradScaler (bf16 на T4 нет). Gradient checkpointing,
чтобы 0.5B вместе с Adam поместилась в 15 ГБ.

In [ ]:
config = AutoConfig.from_pretrained(INIT_MODEL, num_labels=1)
config.classifier_dropout = 0.0
model = AutoModelForTokenClassification.from_pretrained(INIT_MODEL, config=config, dtype=torch.float32)
# По умолчанию перед головой стоит dropout 0.1; для регрессии он только добавляет шум к таргету
for m in model.modules():
    if isinstance(m, torch.nn.Dropout):
        m.p = 0.0
# Голову инициализируем нулями: стартовое предсказание V=0 (средний reward), MSE начинается с ~1, а не с ~20,
# и случайная голова не гонит в backbone мусорные градиенты на первых шагах
with torch.no_grad():
    for n, p in model.named_parameters():
        if n.startswith("score"):
            p.zero_()
model.config.pad_token_id = tok.pad_token_id
model.config.use_cache = False
model.gradient_checkpointing_enable()
model.to(DEVICE)

head = [p for n, p in model.named_parameters() if n.startswith("score")]
assert head, "не нашёл голову score.*"
body = [p for n, p in model.named_parameters() if not n.startswith("score")]
opt = torch.optim.AdamW([{"params": body, "lr": LR}, {"params": head, "lr": HEAD_LR}], weight_decay=0.0)
sched = get_cosine_schedule_with_warmup(opt, WARMUP, TOTAL_STEPS)
scaler = torch.amp.GradScaler("cuda", init_scale=2 ** 10)  # со стандартных 2**16 первые шаги пропускаются из-за переполнения

RUN_META = {"init_model": INIT_MODEL, "rollouts_cfg": rollout_cfg, "debug": DEBUG, "n_train": len(train_ex), "n_val": len(val_ex),
            "epochs": EPOCHS, "micro_bs": MICRO_BS, "accum": ACCUM, "lr": LR, "head_lr": HEAD_LR, "warmup": WARMUP, "seed": SEED}

def save_ckpt(step):
    # Сначала пишем во временную папку, потом подменяем: если сессия упадёт во время записи, старый чекпоинт уцелеет
    tmp = CKPT + "_tmp"
    shutil.rmtree(tmp, ignore_errors=True)
    os.makedirs(tmp)
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                "scaler": scaler.state_dict(), "step": step, "meta": RUN_META}, f"{tmp}/state.pt")
    shutil.rmtree(CKPT, ignore_errors=True)
    os.rename(tmp, CKPT)

def find_ckpt():
    cands = [f"{CKPT}/state.pt"] + sorted(glob.glob(f"/kaggle/input/**/{os.path.basename(OUT)}/ckpt/state.pt", recursive=True))
    return next((c for c in cands if os.path.exists(c)), None)

start_step = 0
if os.path.exists(f"{FINAL}/scorer_config.json"):
    start_step = TOTAL_STEPS
    print("Обучение уже завершено:", FINAL)
elif (path := find_ckpt()):
    st = torch.load(path, map_location=DEVICE, weights_only=False)
    assert st["meta"] == RUN_META, f"Чекпоинт {path} от прогона с другими параметрами: {st['meta']}"
    model.load_state_dict(st["model"]); opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"]); scaler.load_state_dict(st["scaler"])
    start_step = st["step"]
    del st
    print(f"Продолжаю с шага {start_step}/{TOTAL_STEPS} из {path}")
    if not path.startswith(CKPT) and os.path.exists(path.replace("ckpt/state.pt", "metrics.jsonl")):
        shutil.copy(path.replace("ckpt/state.pt", "metrics.jsonl"), METRICS)
elif os.path.exists(METRICS):
    os.remove(METRICS)  # лог от прогона без чекпоинта начинаем заново

def log(rec):
    with open(METRICS, "a") as f:
        f.write(json.dumps(rec) + "\n")

## Валидация

In [ ]:
@torch.no_grad()
def predict_values(examples, bs=16, desc="eval"):
    # Для каждой последовательности возвращает массив из T+1 значений: V после 0..T токенов ответа
    was_training = model.training
    model.eval()
    vals = [None] * len(examples)
    order = sorted(range(len(examples)), key=lambda i: len(examples[i]["ids"]))
    for k in tqdm(range(0, len(order), bs), desc=desc, leave=False):
        idxs = order[k:k + bs]
        b = collate([examples[i] for i in idxs])
        with torch.autocast("cuda", dtype=torch.float16):
            v = model(input_ids=b["input_ids"].to(DEVICE), attention_mask=b["attention_mask"].to(DEVICE)).logits.squeeze(-1).float().cpu().numpy()
        for r, i in enumerate(idxs):
            x = examples[i]
            vals[i] = v[r, x["p"] - 1:len(x["ids"])]
    model.train(was_training)
    return vals

by_pid = defaultdict(list)
for i, x in enumerate(val_ex):
    by_pid[x["pid"]].append(i)
VAL_PAIRS = [(a, b) for ids in by_pid.values() for a, b in combinations(ids, 2) if abs(val_ex[a]["reward"] - val_ex[b]["reward"]) > PAIR_MIN_GAP]
print("val pairs:", len(VAL_PAIRS))

def corr(a, b):
    a, b = np.asarray(a), np.asarray(b)
    return float(np.corrcoef(a, b)[0, 1]) if a.std() > 0 and b.std() > 0 else float("nan")  # до обучения V константа

def evaluate():
    vals = predict_values(val_ex)
    tg = np.array([x["target"] for x in val_ex])
    res = {"val_mse": float(np.mean(np.concatenate([(v - t) ** 2 for v, t in zip(vals, tg)]))),
           "corr@0": corr([v[0] for v in vals], tg),       # насколько reward предсказуем по одному промпту
           "corr@full": corr([v[-1] for v in vals], tg)}
    for t in EVAL_PREFIXES:
        res[f"pair_acc@{t}"] = float(np.mean([(vals[a][min(t, len(vals[a]) - 1)] > vals[b][min(t, len(vals[b]) - 1)]) == (tg[a] > tg[b])
                                              for a, b in VAL_PAIRS]))
    return res

def fmt(res):
    return " ".join(f"{k}={v:.3f}" for k, v in res.items())

## Обучение

In [ ]:
if start_step == 0:
    res = evaluate()
    log({"step": 0, **res})
    print("до обучения:", fmt(res))

model.train()
pbar = tqdm(total=TOTAL_STEPS, initial=start_step, desc="train", unit="step")
ema = None
n_skipped = 0
t0 = time.time()
for step in range(start_step, TOTAL_STEPS):
    loss_acc = 0.0
    for m in range(ACCUM):
        b = {k: v.to(DEVICE) for k, v in collate([train_ex[j] for j in schedule[step * ACCUM + m]]).items()}
        with torch.autocast("cuda", dtype=torch.float16):
            v = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).logits.squeeze(-1)
        loss = ((v.float() - b["target"][:, None]) ** 2)[b["mask"]].mean()
        scaler.scale(loss / ACCUM).backward()
        loss_acc += loss.item() / ACCUM
    scaler.unscale_(opt)
    gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM).item()
    scale = scaler.get_scale()
    scaler.step(opt)
    scaler.update()
    opt.zero_grad(set_to_none=True)
    skipped = scaler.get_scale() < scale  # GradScaler пропустил шаг из-за inf/nan в градиентах
    n_skipped += skipped
    if not skipped:
        sched.step()

    ema = loss_acc if ema is None else 0.95 * ema + 0.05 * loss_acc
    pbar.update(1)
    pbar.set_postfix(loss=f"{ema:.3f}", gnorm=f"{gnorm:.2f}", skipped=n_skipped)
    if (step + 1) % LOG_EVERY == 0:
        log({"step": step + 1, "train_loss": loss_acc, "gnorm": gnorm, "lr": sched.get_last_lr()[0], "sec": time.time() - t0})
    if (step + 1) % EVAL_EVERY == 0 or step + 1 == TOTAL_STEPS:
        res = evaluate()
        log({"step": step + 1, **res})
        pbar.write(f"step {step + 1}: {fmt(res)}")
    if (step + 1) % SAVE_EVERY == 0 or step + 1 == TOTAL_STEPS:
        save_ckpt(step + 1)
pbar.close()

## Финальная модель

Сохраняем веса в fp16 (около 1 ГБ) и `scorer_config.json`: в `03_eval` из него берутся R_MEAN, R_STD и
соглашение о позициях. Полный чекпоинт с оптимизатором весит около 6 ГБ. По умолчанию после сохранения
финальной модели он удаляется, чтобы Output-датасет был лёгким.

In [ ]:
KEEP_CKPT = False

if not os.path.exists(f"{FINAL}/scorer_config.json"):
    final_res = evaluate()
    model.to(torch.float16).save_pretrained(FINAL)
    tok.save_pretrained(FINAL)
    json.dump({"init_model": INIT_MODEL, "reward_mean": R_MEAN, "reward_std": R_STD, "rollouts_cfg": rollout_cfg,
               "value_position": "logits[P-1+t] = V(x, y^t), P = len(prompt_ids), t = число токенов ответа",
               "train_meta": RUN_META, "val": final_res}, open(f"{FINAL}/scorer_config.json", "w"), indent=2)
    print("final:", fmt(final_res))
if not KEEP_CKPT:
    shutil.rmtree(CKPT, ignore_errors=True)
print(os.listdir(FINAL))

## Что смотреть

- `train_loss` и `val_mse` в стандартизованных единицах: 1.0 соответствует константному предсказанию.
  Падение до `1 − corr@0²` значит, что модель выучила только «насколько хорош промпт».
- `pair_acc@t` по t: 0.5 — монетка. Если на 32 токенах accuracy едва выше 0.5, блочный CD с M=32 почти
  ничего не выиграет, и тогда стоит смотреть на M=64 или 128 (в статье большие блоки тоже работали лучше).
- `pair_acc@256` (полный ответ) — потолок: насколько scorer воспроизводит RM, видя ответ целиком.
  В статье похожая проверка (Table 1): классификаторы CD дают 0.63 против 0.71 у RM.

In [ ]:
import matplotlib.pyplot as plt

recs = [json.loads(l) for l in open(METRICS)]
tr = pd.DataFrame([r for r in recs if "train_loss" in r])
ev = pd.DataFrame([r for r in recs if "val_mse" in r]).drop_duplicates("step", keep="last")

fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
if len(tr):
    ax[0].plot(tr.step, tr.train_loss, alpha=0.4, label="train")
ax[0].plot(ev.step, ev.val_mse, "o-", label="val"); ax[0].set_title("MSE (std units)"); ax[0].legend()
for t in EVAL_PREFIXES:
    ax[1].plot(ev.step, ev[f"pair_acc@{t}"], label=f"t={t}")
ax[1].axhline(0.5, color="gray", ls="--"); ax[1].set_title("pairwise acc"); ax[1].legend(fontsize=7)
last = ev.iloc[-1]
ax[2].plot(EVAL_PREFIXES, [last[f"pair_acc@{t}"] for t in EVAL_PREFIXES], "o-")
ax[2].axhline(0.5, color="gray", ls="--"); ax[2].set_xscale("log", base=2); ax[2].set_xlabel("prefix tokens t")
ax[2].set_title(f"pairwise acc vs prefix (step {int(last.step)})")
plt.tight_layout(); plt.show()
print(ev.tail(1).T)

## Сохранение артефактов в Kaggle Dataset

`/kaggle/working` исчезает вместе с сессией, а Output сохраняется только при Save Version. Эта ячейка
сама загружает `scorer/final` и `metrics.jsonl` в приватный датасет через Kaggle API: при первом запуске
создаёт его, при следующих добавляет новую версию. В `03_eval` этот датасет подключается через Add Input.

Нужен секрет `KAGGLE_API_TOKEN` (Add-ons → Secrets), как в `01_rollouts`.

Если сессию нужно остановить посреди обучения, сохрани чекпоинт вручную: `upload_dir(CKPT, SLUG + "-ckpt")`.
Это около 6 ГБ. Потом подключи этот датасет как input, и обучение продолжится с того же шага.

In [ ]:
!pip install -q -U kaggle

import subprocess
from kaggle_secrets import UserSecretsClient
os.environ["KAGGLE_API_TOKEN"] = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")

USERNAME = "karimvafin"
SLUG = "cd-hh-scorer-debug" if DEBUG else "cd-hh-scorer"

def kaggle_cli(*args):
    r = subprocess.run(["kaggle", *args], capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr).strip()

def upload_dir(src, slug, extra_files=(), message="update"):
    # Собираем папку для загрузки: содержимое src плюс extra_files и dataset-metadata.json
    stage = f"{WORK}/upload_{slug}"
    shutil.rmtree(stage, ignore_errors=True)
    shutil.copytree(src, stage)
    for f in extra_files:
        shutil.copy(f, stage)
    ds_id = f"{USERNAME}/{slug}"
    json.dump({"title": slug, "id": ds_id, "licenses": [{"name": "CC0-1.0"}]}, open(f"{stage}/dataset-metadata.json", "w"))
    exists = kaggle_cli("datasets", "status", ds_id)[0] == 0
    code, out = kaggle_cli("datasets", "version", "-p", stage, "-m", message) if exists else kaggle_cli("datasets", "create", "-p", stage)
    print(out[-2000:])
    assert code == 0, "загрузка не удалась, см. вывод выше"
    shutil.rmtree(stage, ignore_errors=True)
    print(f"{'новая версия' if exists else 'создан'}: https://www.kaggle.com/datasets/{ds_id}")

assert os.path.exists(f"{FINAL}/scorer_config.json"), "финальной модели нет: обучение не завершилось"
upload_dir(FINAL, SLUG, extra_files=[METRICS], message=f"scorer final, val: {fmt(json.load(open(f'{FINAL}/scorer_config.json'))['val'])}")